# Game-Level Predictions for Starting Pitchers

This notebook predicts full game statlines (K, BB, H, HR, **Expected Runs**) for starting pitchers.

**Process:**
1. Get scheduled games and lineups from MLB API
2. Estimate batters faced using rolling 5-start average
3. Sum PA-level probability predictions across expected plate appearances
4. Convert to expected runs using linear weights

**Key insight:** All stats are probabilistic sums, not hard predictions. A pitcher might have:
- Expected K: 6.2 (sum of P(K) across ~24 batters faced)
- Expected ER: 3.1 (using run values: 1B=0.45, 2B=0.75, 3B=1.05, HR=1.40, BB=0.30)

**Requirements:**
- Trained FLAML model (`models/flaml_trainer.pkl`)
- Pitcher/batter profiles (`data/profiles/`)

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta

from src.data.mlb_api import (
    get_games_with_lineups,
    check_lineup_status,
    get_pitcher_game_logs,
)
from src.game_predictor import GamePredictor, format_prediction_summary

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

/Applications/miniconda3/envs/mlbenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Select Date and Check Lineup Availability

In [2]:
# Set the date for predictions
# Use today's date or specify a date
# PREDICTION_DATE = date.today().strftime("%Y-%m-%d")
PREDICTION_DATE = "2026-04-01"  # Or specify a date

print(f"Prediction date: {PREDICTION_DATE}")

Prediction date: 2026-04-01


In [3]:
# Check lineup availability
status = check_lineup_status(PREDICTION_DATE)

print(f"Total games: {status['total_games']}")
print(f"Games with lineups: {status['games_with_lineups']}")
print(f"Games with probable pitchers: {status['games_with_probable_pitchers']}")
print(f"\nAll lineups available: {status['all_lineups_available']}")

Total games: 15
Games with lineups: 15
Games with probable pitchers: 15

All lineups available: True


In [4]:
# Show games and their status
print("\nGame Status:")
print("-" * 60)
for game in status['games']:
    away = game['away_team']['abbreviation']
    home = game['home_team']['abbreviation']
    
    away_p = game['away_pitcher']['pitcher_name'] if game['away_pitcher'] else 'TBD'
    home_p = game['home_pitcher']['pitcher_name'] if game['home_pitcher'] else 'TBD'
    
    lineup_status = "Lineups Ready" if game['lineups_available'] else "Awaiting Lineups"
    
    print(f"{away:4s} @ {home:4s} | {away_p:20s} vs {home_p:20s} | {lineup_status}")


Game Status:
------------------------------------------------------------
ATH  @ ATL  | Luis Severino        vs Chris Sale           | Lineups Ready
TEX  @ BAL  | Nathan Eovaldi       vs Trevor Rogers        | Lineups Ready
PIT  @ CIN  | Paul Skenes          vs Andrew Abbott        | Lineups Ready
WSH  @ PHI  | Cade Cavalli         vs Cristopher Sánchez   | Lineups Ready
COL  @ TOR  | Kyle Freeland        vs Kevin Gausman        | Lineups Ready
CWS  @ MIA  | Shane Smith          vs Sandy Alcantara      | Lineups Ready
NYM  @ STL  | Freddy Peralta       vs Matthew Liberatore   | Lineups Ready
TB   @ MIL  | Drew Rasmussen       vs Jacob Misiorowski    | Lineups Ready
BOS  @ HOU  | Garrett Crochet      vs Mike Burrows         | Lineups Ready
LAA  @ CHC  | Yusei Kikuchi        vs Matthew Boyd         | Lineups Ready
DET  @ AZ   | Tarik Skubal         vs Zac Gallen           | Lineups Ready
SF   @ SD   | Adrian Houser        vs Nick Pivetta         | Lineups Ready
NYY  @ SEA  | Cam Schlitt

## 2. Initialize Predictor

In [5]:
# Initialize game predictor
predictor = GamePredictor(
    trainer_path='../models/flaml_trainer.pkl',
    preprocessor_path='../models/matchup_preprocessor.pkl',
    pitcher_profiles_path='../data/profiles/pitcher_arsenal.csv',
    batter_profiles_path='../data/profiles/batter_profiles.csv',
    rolling_starts=5,  # Use last 5 starts for BF average
)

print(f"Predictor initialized")
print(f"Outcome classes: {predictor.outcome_classes}")
print(f"Features: {len(predictor.feature_names)}")

Predictor initialized
Outcome classes: [np.str_('1B'), np.str_('2B'), np.str_('3B'), np.str_('BB'), np.str_('HR'), np.str_('K'), np.str_('OUT')]
Features: 207


## 3. Run Predictions

In [6]:
# Run predictions for all games
predictions = predictor.predict_day(PREDICTION_DATE)

print(f"Predicted {len(predictions)} games")
print(f"Complete: {sum(1 for p in predictions if p['status'] == 'complete')}")
print(f"Partial: {sum(1 for p in predictions if p['status'] == 'partial')}")
print(f"Awaiting lineups: {sum(1 for p in predictions if p['status'] == 'awaiting_lineups')}")
print(f"Failed: {sum(1 for p in predictions if p['status'] == 'failed')}")

Predicted 15 games
Complete: 15
Partial: 0
Awaiting lineups: 0
Failed: 0


In [7]:
# Display predictions
for game in predictions:
    print(f"\n{'='*70}")
    print(f"{game['away_team']} @ {game['home_team']} | Status: {game['status']}")
    print(f"{'='*70}")
    
    if game['away_prediction']:
        print(f"\n[AWAY] {format_prediction_summary(game['away_prediction'])}")
    else:
        print(f"\n[AWAY] No prediction (missing lineup or pitcher data)")
        
    if game['home_prediction']:
        print(f"\n[HOME] {format_prediction_summary(game['home_prediction'])}")
    else:
        print(f"\n[HOME] No prediction (missing lineup or pitcher data)")
    
    if game['errors']:
        print(f"\nErrors: {game['errors']}")


ATH @ ATL | Status: complete

[AWAY] Luis Severino
  Expected BF: 22.0, IP~: 5.0
  K: 4.0, BB: 1.6, H: 5.2, HR: 0.9
  Expected Runs: 2.67
  ERA (this start): 4.76

[HOME] Chris Sale
  Expected BF: 23.0, IP~: 5.5
  K: 6.5, BB: 1.4, H: 5.0, HR: 0.6
  Expected Runs: 2.35
  ERA (this start): 3.83

TEX @ BAL | Status: complete

[AWAY] Nathan Eovaldi
  Expected BF: 24.0, IP~: 5.8
  K: 6.3, BB: 1.6, H: 5.1, HR: 0.8
  Expected Runs: 2.58
  ERA (this start): 4.03

[HOME] Trevor Rogers
  Expected BF: 22.0, IP~: 5.1
  K: 5.5, BB: 2.0, H: 4.8, HR: 0.7
  Expected Runs: 2.49
  ERA (this start): 4.42

PIT @ CIN | Status: complete

[AWAY] Paul Skenes
  Expected BF: 20.0, IP~: 4.7
  K: 6.0, BB: 1.8, H: 4.2, HR: 0.6
  Expected Runs: 2.13
  ERA (this start): 4.09

[HOME] Andrew Abbott
  Expected BF: 22.0, IP~: 5.2
  K: 5.0, BB: 1.7, H: 4.8, HR: 0.6
  Expected Runs: 2.38
  ERA (this start): 4.16

WSH @ PHI | Status: complete

[AWAY] Cade Cavalli
  Expected BF: 21.0, IP~: 4.7
  K: 4.6, BB: 1.6, H: 5.2, HR

## 4. Summary Table

In [8]:
# Build summary dataframe
rows = []
for game in predictions:
    for side, pred_key in [('Away', 'away_prediction'), ('Home', 'home_prediction')]:
        pred = game[pred_key]
        if pred:
            stats = pred['expected_stats']
            rows.append({
                'Game': f"{game['away_team']} @ {game['home_team']}",
                'Side': side,
                'Pitcher': pred['pitcher_name'],
                'BF': pred['expected_bf'],
                'IP~': stats['IP_approx'],
                'K': stats['K'],
                'BB': stats['BB'],
                'H': stats['H'],
                'HR': stats['HR'],
                'ER': stats['ER'],  # Expected runs
            })

if rows:
    summary_df = pd.DataFrame(rows)
    print("\nPrediction Summary (all values are expected/probabilistic):")
    display(summary_df)
else:
    print("No predictions available (likely waiting for lineups)")


Prediction Summary (all values are expected/probabilistic):


,Game,Side,Pitcher,BF,IP~,K,BB,H,HR,ER
0,ATH @ ATL,Away,Luis Severino,22.0,5.0,4.03,1.60,5.25,0.89,2.67
1,ATH @ ATL,Home,Chris Sale,23.0,5.5,6.52,1.41,4.99,0.57,2.35
2,TEX @ BAL,Away,Nathan Eovaldi,24.0,5.8,6.27,1.61,5.11,0.76,2.58
3,TEX @ BAL,Home,Trevor Rogers,22.0,5.1,5.46,1.96,4.81,0.66,2.49
4,PIT @ CIN,Away,Paul Skenes,20.0,4.7,5.99,1.79,4.16,0.57,2.13
5,PIT @ CIN,Home,Andrew Abbott,22.0,5.2,4.97,1.68,4.85,0.61,2.38
6,WSH @ PHI,Away,Cade Cavalli,21.0,4.7,4.60,1.63,5.24,0.75,2.59
7,WSH @ PHI,Home,Cristopher Sánchez,24.0,5.8,5.78,1.26,5.37,0.35,2.28
8,COL @ TOR,Away,Kyle Freeland,22.0,5.1,4.03,1.30,5.29,0.50,2.35
9,COL @ TOR,Home,Kevin Gausman,21.0,5.2,5.83,1.08,4.46,0.54,2.12


## 5. Detailed View: Single Game

In [9]:
# Select a game for detailed view (change index as needed)
GAME_INDEX = 0

if predictions and len(predictions) > GAME_INDEX:
    game = predictions[GAME_INDEX]
    print(f"Detailed view: {game['away_team']} @ {game['home_team']}")
else:
    print("No games available")

Detailed view: ATH @ ATL


In [10]:
# Show batter-by-batter breakdown for away pitcher
if predictions and predictions[GAME_INDEX].get('away_prediction'):
    pred = predictions[GAME_INDEX]['away_prediction']
    print(f"\n{pred['pitcher_name']} vs opposing lineup:\n")
    
    batter_rows = []
    for bp in pred['batter_predictions']:
        probs = bp['probabilities']
        batter_rows.append({
            'Order': bp['batting_order'],
            'TTO': bp['times_through'],
            'Batter': bp['batter_name'][:20],
            'P(K)': f"{probs['K']:.1%}",
            'P(BB)': f"{probs['BB']:.1%}",
            'P(H)': f"{(probs['1B']+probs['2B']+probs['3B']+probs['HR']):.1%}",
            'P(HR)': f"{probs['HR']:.1%}",
            'P(OUT)': f"{probs['OUT']:.1%}",
        })
    
    display(pd.DataFrame(batter_rows))
else:
    print("No away prediction available")


Luis Severino vs opposing lineup:



,Order,TTO,Batter,P(K),P(BB),P(H),P(HR),P(OUT)
0,1,1,Ronald Acuña Jr.,19.1%,8.3%,26.0%,4.7%,46.6%
1,2,1,Drake Baldwin,14.2%,7.1%,25.5%,3.1%,53.2%
2,3,1,Matt Olson,20.8%,10.7%,22.4%,5.6%,46.1%
3,4,1,Austin Riley,21.9%,7.1%,24.0%,4.8%,47.0%
4,5,1,Mike Yastrzemski,19.4%,9.4%,21.3%,5.0%,49.8%
5,6,1,Ozzie Albies,17.3%,5.8%,22.9%,4.1%,54.0%
6,7,1,Michael Harris II,18.9%,4.6%,26.6%,4.0%,49.9%
7,8,1,Dominic Smith,18.6%,7.0%,22.4%,2.5%,51.9%
8,9,1,Mauricio Dubón,13.3%,3.5%,22.4%,1.6%,60.8%
9,1,2,Ronald Acuña Jr.,19.1%,8.3%,26.0%,4.7%,46.6%


## 6. Export Predictions

In [11]:
# Export predictions to CSV
if rows:
    output_path = f"../data/predictions_{PREDICTION_DATE}.csv"
    summary_df.to_csv(output_path, index=False)
    print(f"Saved predictions to {output_path}")
else:
    print("No predictions to export")

Saved predictions to ../data/predictions_2026-04-01.csv


---

## Appendix: Manual Single Pitcher Prediction

Use this to predict a specific pitcher against a specific lineup.

In [12]:
# Example: Manual prediction for a specific pitcher
# Uncomment and modify as needed

# pitcher_id = 592789  # Example: Gerrit Cole
# pitcher_name = "Gerrit Cole"
# p_throws = "R"
# 
# # Define lineup (you can get this from MLB API or enter manually)
# lineup = [
#     {"batter_id": 545361, "batter_name": "Mike Trout", "stand": "R"},
#     {"batter_id": 660271, "batter_name": "Shohei Ohtani", "stand": "L"},
#     # ... add more batters
# ]
# 
# prediction = predictor.predict_game(
#     pitcher_id=pitcher_id,
#     pitcher_name=pitcher_name,
#     p_throws=p_throws,
#     lineup=lineup,
#     season=2024,
#     expected_bf=25,  # Or let it use rolling average
# )
# 
# print(format_prediction_summary(prediction))

## Appendix: Check Pitcher's Recent Starts

In [13]:
# Look up a pitcher's recent game logs
# PITCHER_ID = 592789  # Example: Gerrit Cole
# 
# logs = get_pitcher_game_logs(PITCHER_ID, limit=10)
# 
# print(f"Recent starts:")
# for log in logs:
#     print(f"  {log['date']}: {log['innings_pitched']} IP, {log['batters_faced']} BF, "
#           f"{log['strikeouts']} K, {log['walks']} BB")